# Tags — Topic Tag Generation

Runs tag generation across the vault using the configured backend and compares
results against the ground truth in `data/ground_truth.yaml`.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, '../src')

from glob import glob
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

from notemine.config import load_config
from notemine.frontmatter_io import load_note
from notemine.ground_truth import load_ground_truth
from notemine.tasks import run_tags

load_dotenv('../.env')
config = load_config('../config.toml')
gt = load_ground_truth('../' + config['vault']['ground_truth'].lstrip('./'))

vault = '../' + config['vault']['path'].lstrip('./')
config['vault']['prompts_dir'] = '../' + config['vault']['prompts_dir'].lstrip('./')

notes = sorted(
    p for p in glob(f'{vault}/**/*.md', recursive=True)
    if not Path(p).name.startswith('index_')
)
print(f'{len(notes)} notes found')

10 notes found


## Run tag generation

In [2]:
import time

backend = config['run']['backend']

results = []
for path in notes:
    name = Path(path).name
    post = load_note(path)
    if len(post.content.strip()) < 50:
        continue

    gt_entry = gt.get(name, {})
    gt_tags = set(gt_entry.get('tags', []))

    row = {'file': name, 'lang': gt_entry.get('lang', '?'), 'ground_truth': sorted(gt_tags)}

    t0 = time.perf_counter()
    try:
        row[backend] = run_tags(backend, post.content, config)
    except Exception as e:
        row[backend] = f'ERROR: {e}'
    row['elapsed_s'] = time.perf_counter() - t0

    results.append(row)
    print(f'  {name}  ({row["elapsed_s"]:.1f}s)')

print(f'\nDone — {len(results)} notes processed')

  long_ai_healthcare_oncology_research_en.md  (4.9s)
  long_interview_climate_scientist_sea_level_en.md  (2.1s)
  long_opinion_eu_ai_act_digital_policy_en.md  (2.1s)
  medium_climate_blog_paris_agreement_en.md  (1.9s)
  medium_history_dutch_golden_age_en.md  (1.9s)
  medium_travel_guide_amsterdam_en.md  (2.0s)
  short_recipe_dutch_stamppot_en.md  (1.7s)
  short_review_samsung_galaxy_s25_ultra_en.md  (1.5s)
  short_sports_news_ajax_champions_league_en.md  (1.7s)
  short_tech_news_openai_announcement_en.md  (1.5s)

Done — 10 notes processed


In [ ]:
import json

out_dir = Path('../results/tags')
out_dir.mkdir(parents=True, exist_ok=True)

for r in results:
    stem = Path(r['file']).stem
    payload = r[backend]
    data = {
        'file': r['file'],
        'backend': backend,
        'model': config[backend]['model'],
        'elapsed_s': round(r['elapsed_s'], 3),
        'ground_truth': r['ground_truth'],
        'tags': payload if not isinstance(payload, str) else [],
        'error': payload if isinstance(payload, str) else None,
    }
    (out_dir / f'{stem}.json').write_text(json.dumps(data, ensure_ascii=False, indent=2))

print(f'Saved {len(results)} files to {out_dir}')

## Summary comparison

In [4]:
def tag_stats(gt_tags, pred):
    if isinstance(pred, str):
        return pred
    gt_set = set(gt_tags)
    pred_set = set(pred)
    matched = len(gt_set & pred_set)
    extra = len(pred_set - gt_set)
    pct = int(100 * matched / len(gt_set)) if gt_set else 0
    return f'{matched}/{len(gt_set)} ({pct}%)  +{extra} extra'

rows = []
for r in results:
    rows.append({
        'file': r['file'],
        'lang': r['lang'],
        'ground_truth': r['ground_truth'],
        backend: tag_stats(r['ground_truth'], r[backend]),
    })

pd.set_option('display.max_colwidth', 40)
pd.DataFrame(rows)

,file,lang,ground_truth,ollama
0,long_ai_healthcare_oncology_research...,en,"[ai, amsterdam-umc, cancer-diagnosis...",2/9 (22%) +4 extra
1,long_interview_climate_scientist_sea...,en,"[climate-science, delta-works, delta...",2/9 (22%) +3 extra
2,long_opinion_eu_ai_act_digital_polic...,en,"[big-tech, chatgpt, digital-policy, ...",1/9 (11%) +4 extra
3,medium_climate_blog_paris_agreement_...,en,"[climate-change, cop30, emissions, f...",2/8 (25%) +3 extra
4,medium_history_dutch_golden_age_en.md,en,"[17th-century, amsterdam, colonialis...",0/8 (0%) +5 extra
5,medium_travel_guide_amsterdam_en.md,en,"[amsterdam, canal, cycling, food, jo...",5/9 (55%) +3 extra
6,short_recipe_dutch_stamppot_en.md,en,"[boerenkool, comfort-food, dutch-cui...",1/7 (14%) +4 extra
7,short_review_samsung_galaxy_s25_ultr...,en,"[android, camera, galaxy-s25-ultra, ...",0/7 (0%) +5 extra
8,short_sports_news_ajax_champions_lea...,en,"[ajax, amsterdam, bayernmunich, cham...",4/7 (57%) +3 extra
9,short_tech_news_openai_announcement_...,en,"[ai, gpt-5, llm, microsoft, openai, ...",0/7 (0%) +5 extra


## Inspect a single note

Change `INSPECT` to any filename to see the full tag lists side by side.

In [5]:
INSPECT = Path(notes[0]).name

row = next((r for r in results if r['file'] == INSPECT), None)
if row is None:
    print(f'{INSPECT} not found')
else:
    gt_set = set(row['ground_truth'])
    pred = set(row[backend]) if not isinstance(row[backend], str) else set()

    all_tags = sorted(gt_set | pred)
    tag_rows = [{
        'tag': t,
        'ground_truth': '✓' if t in gt_set else '',
        backend: '✓' if t in pred else '',
    } for t in all_tags]

    print(f'=== {INSPECT} ===')
    display(pd.DataFrame(tag_rows))

=== long_ai_healthcare_oncology_research_en.md ===


,tag,ground_truth,ollama
0,ai,✓,
1,amsterdam-umc,✓,
2,artificial-intelligence-assisted,,✓
3,cancer-diagnosis,✓,
4,clinical-trial,✓,
5,colorectal-cancer,,✓
6,deep-learning,✓,
7,healthcare,✓,
8,machine-learning,✓,✓
9,medical-imaging,,✓


## (Optional) Save results to frontmatter

Uncomment to write results under `notemine.tags.<backend>`.

In [6]:
# from notemine.frontmatter_io import save_note
#
# for path in notes:
#     name = Path(path).name
#     row = next((r for r in results if r['file'] == name), None)
#     if row is None:
#         continue
#     post = load_note(path)
#     notemine = post.metadata.get('notemine', {})
#     tags_data = notemine.get('tags', {})
#     if not isinstance(row.get(backend), str):
#         tags_data[backend] = row[backend]
#     notemine['tags'] = tags_data
#     post.metadata['notemine'] = notemine
#     save_note(path, post)
# print('Saved.')